# Практическая работа № 3. Численные методы решения систем линейных уравнений

Вариант: 9

**Цель:** найти решение СЛАУ прямым методом Гаусса, методом простой итерации и проверить результат инструментальным средством.

**Вариант 9:**

$$\begin{aligned}
-9{,}11x_1+1{,}02x_2-0{,}73x_3&=-1{,}25,\\
7{,}61x_1+6{,}25x_2-2{,}32x_3&=2{,}33,\\
-4{,}64x_1+1{,}13x_2-8{,}88x_3&=-3{,}75.
\end{aligned}$$

In [1]:
A = [
    [-9.11, 1.02, -0.73],
    [7.61, 6.25, -2.32],
    [-4.64, 1.13, -8.88],
]

B = [-1.25, 2.33, -3.75]

## Задание 1. Метод Гаусса

Схема единственного деления с округлением промежуточных чисел до 3 знаков, контрольные суммы и обратный ход. Также выполнен расчёт без промежуточного округления.

In [2]:
from decimal import Decimal, ROUND_HALF_UP

def r3(v):
    """Округление до трёх знаков после запятой (как на калькуляторе)."""
    return float(Decimal(str(v)).quantize(Decimal("0.001"), rounding=ROUND_HALF_UP))

def fmt(row):
    return " | ".join(f"{v:9.3f}" for v in row)

def gauss_manual(A, B):
    n = len(A)
    # Расширенная матрица + столбец контрольных сумм
    M = [list(A[i]) + [B[i], r3(sum(A[i]) + B[i])] for i in range(n)]
    leads = []
    print("Раздел A (исходная система; последний столбец — Σ):")
    for row in M:
        print("   ", fmt(row))
    pivots_rows = []
    cur = M
    for k in range(n):
        lead = cur[0][k]
        leads.append(lead)
        prow = [r3(v / lead) if j >= k else 0.0 for j, v in enumerate(cur[0])]
        prow[k] = 1.0
        ctrl = r3(sum(prow[k:-1]))
        print(f"  ведущий элемент {lead:.3f}; строка / ведущий:")
        print("   ", fmt(prow), f"   контроль: Σ строки = {ctrl:.3f}")
        pivots_rows.append(prow)
        nxt = []
        for row in cur[1:]:
            pv = row[k]
            new = [r3(row[j] - pv * prow[j]) if j > k else 0.0 for j in range(len(row))]
            nxt.append(new)
        if nxt:
            print(f"Раздел A{k + 1}:")
            for row in nxt:
                print("   ", fmt(row), f"   контроль: Σ строки = {r3(sum(row[k + 1:-1])):.3f}")
        cur = nxt
    # Обратный ход
    x = [0.0] * n
    xs = [0.0] * n  # решение для столбца Σ (должно быть x + 1)
    for i in reversed(range(n)):
        row = pivots_rows[i]
        x[i] = r3(row[n] - sum(row[j] * x[j] for j in range(i + 1, n)))
        xs[i] = r3(row[n + 1] - sum(row[j] * xs[j] for j in range(i + 1, n)))
    print("Раздел B (обратный ход):")
    for i in range(n):
        print(f"    x{i + 1} = {x[i]:8.3f}   контроль x̄{i + 1} = {xs[i]:8.3f} (должно быть x + 1 = {x[i] + 1:.3f})")
    det = 1.0
    for l in leads:
        det *= l
    return x, leads, det

def gauss_exact(A, B):
    n = len(A)
    M = [list(A[i]) + [B[i]] for i in range(n)]
    det = 1.0
    for k in range(n):
        lead = M[k][k]
        det *= lead
        M[k] = [v / lead for v in M[k]]
        for i in range(k + 1, n):
            f = M[i][k]
            M[i] = [M[i][j] - f * M[k][j] for j in range(n + 1)]
    x = [0.0] * n
    for i in reversed(range(n)):
        x[i] = M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n))
    return x, det

def residuals(x):
    return [B[i] - sum(A[i][j] * x[j] for j in range(3)) for i in range(3)]

print("=== а) Ручная схема единственного деления (3 знака) ===")
x, leads, det = gauss_manual(A, B)
print("Подстановка в исходную систему и невязки:")
for i in range(3):
    s = sum(A[i][j] * x[j] for j in range(3))
    print(f"    ур.{i + 1}: левая часть = {s:.4f}, b = {B[i]:.2f}, невязка = {B[i] - s:+.4f}")
print("Ведущие элементы:", ", ".join(f"{l:.3f}" for l in leads))
print(f"Определитель D = {' * '.join(f'({l:.3f})' for l in leads)} = {det:.3f}")

print("\n=== б) Программа (без промежуточного округления) ===")
xe, de = gauss_exact(A, B)
print("x =", ", ".join(f"{v:.6f}" for v in xe))
print(f"D = {de:.6f}")
print("Невязки:", ", ".join(f"{v:.1e}" for v in residuals(xe)))

=== а) Ручная схема единственного деления (3 знака) ===
Раздел A (исходная система; последний столбец — Σ):
       -9.110 |     1.020 |    -0.730 |    -1.250 |   -10.070
        7.610 |     6.250 |    -2.320 |     2.330 |    13.870
       -4.640 |     1.130 |    -8.880 |    -3.750 |   -16.140
  ведущий элемент -9.110; строка / ведущий:
        1.000 |    -0.112 |     0.080 |     0.137 |     1.105    контроль: Σ строки = 1.105
Раздел A1:
        0.000 |     7.102 |    -2.929 |     1.287 |     5.461    контроль: Σ строки = 5.460
        0.000 |     0.610 |    -8.509 |    -3.114 |   -11.013    контроль: Σ строки = -11.013
  ведущий элемент 7.102; строка / ведущий:
        0.000 |     1.000 |    -0.412 |     0.181 |     0.769    контроль: Σ строки = 0.769
Раздел A2:
        0.000 |     0.000 |    -8.258 |    -3.224 |   -11.482    контроль: Σ строки = -11.482
  ведущий элемент -8.258; строка / ведущий:
        0.000 |     0.000 |     1.000 |     0.390 |     1.390    контроль: Σ строки = 1.3

## Задание 2. Метод простой итерации

Второе уравнение заменяется суммой первого и второго, чтобы получить диагональное преобладание. Затем приводим систему к виду $x=Cx+d$ и проверяем $q=\|C\|_\infty<1$. Точность $10^{-4}$.

In [3]:
A2 = [A[0][:], [A[0][j] + A[1][j] for j in range(3)], A[2][:]]

B2 = [B[0], B[0] + B[1], B[2]]

def to_iteration_form(A, B):
    n = len(A)
    C = [[0.0 if i == j else -A[i][j] / A[i][i] for j in range(n)] for i in range(n)]
    d = [B[i] / A[i][i] for i in range(n)]
    return C, d

def norms(C):
    n = len(C)
    rows = max(sum(abs(C[i][j]) for j in range(n)) for i in range(n))
    cols = max(sum(abs(C[i][j]) for i in range(n)) for j in range(n))
    eucl = sum(C[i][j] ** 2 for i in range(n) for j in range(n)) ** 0.5
    return rows, cols, eucl

def iterate(C, d, stop, label):
    x = d[:]
    k = 0
    print(f"{label}; порог остановки {stop:.3e}")
    print(f"{'k':>2} | {'x1':>10} | {'x2':>10} | {'x3':>10} | {'max|Δ|':>9}")
    print(f"{k:2d} | {x[0]:10.6f} | {x[1]:10.6f} | {x[2]:10.6f} |")
    while True:
        new = [sum(C[i][j] * x[j] for j in range(3)) + d[i] for i in range(3)]
        k += 1
        diff = max(abs(new[i] - x[i]) for i in range(3))
        print(f"{k:2d} | {new[0]:10.6f} | {new[1]:10.6f} | {new[2]:10.6f} | {diff:9.2e}")
        x = new
        if diff <= stop:
            return x, k

eps = 1e-4
print("Преобразованная система (ур.2 = ур.1 + ур.2):")
for i in range(3):
    print("   ", " ".join(f"{A2[i][j]:+7.2f}x{j + 1}" for j in range(3)), f"= {B2[i]:+.2f}")
C, d = to_iteration_form(A2, B2)
print("Вид x = Cx + d:")
for i in range(3):
    terms = " ".join(f"{C[i][j]:+.4f}x{j + 1}" for j in range(3) if j != i)
    print(f"    x{i + 1} = {d[i]:+.4f} {terms}")
q1, q2, q3 = norms(C)
print(f"Условия сходимости: (3.11) по строкам = {q1:.4f}; (3.12) по столбцам = {q2:.4f}; (3.13) евклидова = {q3:.4f}")
q = q1
print(f"Выбрана метрика rho1, q = {q:.4f} < 1\n")
xa, ka = iterate(C, d, eps * (1 - q) / q, "а) Оценочная формула (3.15)")
print(f"Решение а): x1 = {xa[0]:.4f}, x2 = {xa[1]:.4f}, x3 = {xa[2]:.4f} (итераций {ka})\n")
xb, kb = iterate(C, d, eps, "б) Эмпирический критерий max|Δ| < eps")
print(f"Решение б): x1 = {xb[0]:.4f}, x2 = {xb[1]:.4f}, x3 = {xb[2]:.4f} (итераций {kb})")

Преобразованная система (ур.2 = ур.1 + ур.2):
      -9.11x1   +1.02x2   -0.73x3 = -1.25
      -1.50x1   +7.27x2   -3.05x3 = +1.08
      -4.64x1   +1.13x2   -8.88x3 = -3.75
Вид x = Cx + d:
    x1 = +0.1372 +0.1120x2 -0.0801x3
    x2 = +0.1486 +0.2063x1 +0.4195x3
    x3 = +0.4223 -0.5225x1 +0.1273x2
Условия сходимости: (3.11) по строкам = 0.6498; (3.12) по столбцам = 0.7288; (3.13) евклидова = 0.7258
Выбрана метрика rho1, q = 0.6498 < 1

а) Оценочная формула (3.15); порог остановки 5.390e-05
 k |         x1 |         x2 |         x3 |    max|Δ|
 0 |   0.137212 |   0.148556 |   0.422297 |
 1 |   0.120005 |   0.354034 |   0.369505 |  2.05e-01
 2 |   0.147242 |   0.328335 |   0.404643 |  3.51e-02
 3 |   0.141549 |   0.348697 |   0.387141 |  2.04e-02
 4 |   0.145231 |   0.340180 |   0.392707 |  8.52e-03
 5 |   0.143832 |   0.343274 |   0.389699 |  3.09e-03
 6 |   0.144419 |   0.341724 |   0.390824 |  1.55e-03
 7 |   0.144155 |   0.342317 |   0.390320 |  5.93e-04
 8 |   0.144262 |   0.342051 

## Задание 3. Инструментальная проверка

Сравнение результата с `numpy.linalg.solve` и проверка определителя.

In [4]:
import contextlib, io

import numpy as np

x_np = np.linalg.solve(np.array(A), np.array(B))
det_np = np.linalg.det(np.array(A))
with contextlib.redirect_stdout(io.StringIO()):
    x_man, _, det_man = gauss_manual(A, B)
    x_ex, _ = gauss_exact(A, B)
    C, d = to_iteration_form(A2, B2)
    q = norms(C)[0]
    x_it, _ = iterate(C, d, 1e-4 * (1 - q) / q, "")
print("numpy.linalg.solve :", ", ".join(f"{v:.6f}" for v in x_np), f"; det = {det_np:.4f}")
rows = [("Гаусс, ручная схема (3 знака)", x_man), ("Гаусс, программа", x_ex), ("Простая итерация, 1e-4", x_it)]
for name, x in rows:
    dev = max(abs(x[i] - x_np[i]) for i in range(3))
    print(f"{name:32s}: " + ", ".join(f"{v:.6f}" for v in x) + f"   макс. отклонение {dev:.1e}")

numpy.linalg.solve : 0.144229, 0.342129, 0.390471 ; det = 534.1832
Гаусс, ручная схема (3 знака)   : 0.144000, 0.342000, 0.390000   макс. отклонение 4.7e-04
Гаусс, программа                : 0.144229, 0.342129, 0.390471   макс. отклонение 5.6e-17
Простая итерация, 1e-4          : 0.144235, 0.342115, 0.390482   макс. отклонение 1.4e-05


**Вывод.** Решение: $(x_1,x_2,x_3)\approx(0{,}144229;0{,}342129;0{,}390471)$. Определитель $D\approx 534{,}183167$. Метод Гаусса и NumPy согласуются; метод итераций достигает требуемой точности за 10 шагов.